In [6]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project="arxiv-abstract-classifier")

def pull(start, end, limit=40000):
    sql = f"""
      SELECT abstract, primary_category
      FROM `arxiv-abstract-classifier.arxiv.papers_cs`
      WHERE update_date BETWEEN '{start}' AND '{end}'
        AND abstract IS NOT NULL AND LENGTH(abstract) > 50
      LIMIT {limit}
    """
    rows = client.query(sql).result()
    return pd.DataFrame(
        [{"abstract": r["abstract"], "primary_category": r["primary_category"]} for r in rows]
    )

train_df = pull('2015-01-01', '2018-12-31')
serve_df = pull('2023-01-01', '2025-12-31')
print("train:", train_df.shape, " serve:", serve_df.shape)

train: (35577, 2)  serve: (40000, 2)


In [7]:
MARKERS = ["transformer", "llm", "diffusion", "rag", "attention", "prompt"]
THRESHOLD = 0.05  # L-infinity: flag a feature if its distribution shifts more than this

def featurize(df):
    out = pd.DataFrame()
    out["label"] = df["primary_category"]
    text = df["abstract"].str.lower().fillna("")
    for m in MARKERS:
        out[f"has_{m}"] = text.str.contains(rf"\b{m}\b")
    return out

train_f = featurize(train_df)
serve_f = featurize(serve_df)

def linf(col):
    # max abs difference in category proportions between train and serve
    a = train_f[col].value_counts(normalize=True)
    b = serve_f[col].value_counts(normalize=True)
    idx = a.index.union(b.index)
    return (a.reindex(idx, fill_value=0) - b.reindex(idx, fill_value=0)).abs().max()

print("=== Drift report (L-infinity distance, train 2015-18 vs serve 2023-25) ===\n")
for col in ["label"] + [f"has_{m}" for m in MARKERS]:
    d = linf(col)
    flag = "  <-- DRIFT" if d > THRESHOLD else ""
    print(f"{col:16s}  L-inf = {d:.3f}{flag}")

# the vocabulary-drift story, made concrete:
print("\n=== Marker term prevalence (% of abstracts mentioning it) ===")
for m in MARKERS:
    t = train_f[f"has_{m}"].mean() * 100
    s = serve_f[f"has_{m}"].mean() * 100
    print(f"{m:12s}  2015-18: {t:5.1f}%   2023-25: {s:5.1f}%")

=== Drift report (L-infinity distance, train 2015-18 vs serve 2023-25) ===

label             L-inf = 0.090  <-- DRIFT
has_transformer   L-inf = 0.074  <-- DRIFT
has_llm           L-inf = 0.020
has_diffusion     L-inf = 0.031
has_rag           L-inf = 0.000
has_attention     L-inf = 0.048
has_prompt        L-inf = 0.027

=== Marker term prevalence (% of abstracts mentioning it) ===
transformer   2015-18:   0.3%   2023-25:   7.8%
llm           2015-18:   0.0%   2023-25:   2.0%
diffusion     2015-18:   0.5%   2023-25:   3.6%
rag           2015-18:   0.0%   2023-25:   0.0%
attention     2015-18:   6.2%   2023-25:  11.0%
prompt        2015-18:   0.1%   2023-25:   2.8%


In [9]:
import numpy as np, tensorflow as tf

classes = sorted(train_df["primary_category"].unique())
idx = {c: i for i, c in enumerate(classes)}
X = train_df["abstract"].to_numpy()
y = np.array([idx[c] for c in train_df["primary_category"]])

vec = tf.keras.layers.TextVectorization(max_tokens=20000, output_sequence_length=256)
vec.adapt(tf.data.Dataset.from_tensor_slices(X).batch(256))

model = tf.keras.Sequential([
    tf.keras.Input(shape=(1,), dtype=tf.string), vec,
    tf.keras.layers.Embedding(20000, 128, mask_zero=True),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(len(classes), activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(X, y, validation_split=0.1, batch_size=256, epochs=5)
model.save("arxiv_clf.keras")

Epoch 1/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 13s 89ms/step - accuracy: 0.5972 - loss: 1.0008 - val_accuracy: 0.2619 - val_loss: 4.8217
Epoch 2/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 11s 86ms/step - accuracy: 0.8449 - loss: 0.4461 - val_accuracy: 0.2372 - val_loss: 5.6578
Epoch 3/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 10s 77ms/step - accuracy: 0.8780 - loss: 0.3511 - val_accuracy: 0.2448 - val_loss: 5.9328
Epoch 4/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 10s 78ms/step - accuracy: 0.8927 - loss: 0.3052 - val_accuracy: 0.2470 - val_loss: 5.9448
Epoch 5/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 10s 75ms/step - accuracy: 0.9060 - loss: 0.2709 - val_accuracy: 0.2420 - val_loss: 6.1155
